In [14]:
# ── CASE STUDY 2 STARTER CODE ──────────────────────────────────────────────
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
)
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

In [15]:
# --- Banking chatbot prompt --
banking_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are a helpful banking assistant for CitizenBank.
Always address the customer by name once you know it.
Be professional, concise, and empathetic.
Previous conversation:
{history}
Customer: {input}
CitizenBank Assistant:"""
)

In [16]:
# --- Step 1: Buffer Memory Chain (first 2 turns — KYC capture) --
buffer_memory = ConversationBufferMemory()
# YOUR CODE HERE — create LLMChain with buffer_memory

buffer_chain = LLMChain(llm=llm, prompt=banking_prompt, memory=buffer_memory)
turn1 = buffer_chain.predict(input="Hi, my name is Rajesh and I have a savings account with you.")
print("Turn 1:", turn1)
turn2 = buffer_chain.predict(input="Can you tell me my current account balance?")
print("Turn 2:", turn2)

Turn 1: Hello Rajesh, thank you for reaching out. I see you have a savings account with us. How may I assist you today?
Turn 2: Rajesh, to ensure the security of your account and protect your personal information, I'll need to verify your identity before I can provide your current balance. Could you please provide the last four digits of your Social Security Number or your date of birth?


In [17]:
# --- Step 2: Window Memory Chain (turns 3-5 — active support) --
window_memory = ConversationBufferWindowMemory(k=3)
# YOUR CODE HERE — create LLMChain with window_memory
window_chain = LLMChain(llm=llm, prompt=banking_prompt, memory=window_memory)
turn3 = window_chain.predict(input="I see a transaction of Rs 5000 on March 10 to a merchant I don't recognize. Can you help?")
print("Turn 3:", turn3)
turn4 = window_chain.predict(input="Also, I'd like to temporarily block my debit card for security reasons.")
print("Turn 4:", turn4)
turn5 = window_chain.predict(input="Thank you. How long does a dispute resolution take?")
print("Turn 5:", turn5)

Turn 3: I understand your concern about an unrecognized transaction. I can certainly help you investigate this. To begin, could you please provide your full name so I can securely access your account details?
Turn 4: I understand, and I can certainly help you temporarily block your debit card for security. To proceed with both blocking your card and investigating the unrecognized transaction, please provide your full name.
Turn 5: A dispute resolution typically takes between 30 to 90 days, though this can vary depending on the complexity of the case and the merchant's response time.

To initiate the dispute process and temporarily block your debit card, please provide your full name.


In [18]:
# --- Step 3: Summary Memory (end of session — handoff note) --
summary_memory = ConversationSummaryMemory(llm=llm)
# YOUR CODE HERE — create LLMChain with summary_memory
summary_chain = LLMChain(llm=llm, prompt=banking_prompt, memory=summary_memory)

In [19]:
# Feed the key events into summary memory
summary_chain.predict(input="Summarize this session: Rajesh with savings account, asked for balance, reported a suspicious transaction, requested card block, and inquired about dispute resolution time.")

'Certainly, Rajesh. To summarize our session:\n\nYou inquired about your savings account balance, reported a suspicious transaction, requested a block on your card, and asked about the typical resolution time for disputes.'

In [20]:
print("\n--- HANDOFF SUMMARY FOR HUMAN AGENT ---")
# YOUR CODE HERE — print the summary memory buffer
print(summary_memory.load_memory_variables({})["history"])


--- HANDOFF SUMMARY FOR HUMAN AGENT ---
New summary:
The human asked the AI to summarize a session where Rajesh inquired about his savings account balance, reported a suspicious transaction, requested a card block, and asked about the typical resolution time for disputes. The AI confirmed and summarized these points.
